# 第一模块补充：基于聚类的类别划分

本 Notebook 用 **embedding + HDBSCAN 聚类 + 关键词匹配** 替代 LLM 分类，实现多标签分类。

## 流程
1. 加载清洗后的评论数据
2. 文本向量化（text-embedding-v4）
3. 降维（PCA 50D → 用于聚类）
4. HDBSCAN 聚类 + KMeans 对比
5. 簇 → 类别映射（关键词匹配）
6. 多标签分配（余弦相似度 + 阈值）
7. 与 LLM 分类结果对比评估

**前置条件**：
- 清洗后的数据文件：`data/processed/hotel_comments_cleaned.csv`
- LLM 分类结果：`data/processed/enriched_comments.csv`（用于对比）
- 配置文件：`config/categories.json`
- 环境变量：`DASHSCOPE_API_KEY`

## 环境配置

In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
import jieba
import jieba.analyse

# API 客户端
from dashscope import TextEmbedding

# 尝试导入 HDBSCAN（如未安装则提示）
try:
    import hdbscan
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False
    print("警告: hdbscan 未安装，将仅使用 KMeans。安装: pip install hdbscan")

# 配置
API_KEY = os.getenv("DASHSCOPE_API_KEY")
if not API_KEY:
    raise EnvironmentError("请设置 DASHSCOPE_API_KEY 环境变量")

DATA_DIR = Path("data")
CONFIG_DIR = Path("config")
PROCESSED_DIR = DATA_DIR / "processed"

print("环境配置完成")
print(f"HDBSCAN 可用: {HAS_HDBSCAN}")

## 1. 加载数据与类别配置

In [ ]:
# 加载清洗后的评论数据
df_cleaned = pd.read_csv(PROCESSED_DIR / "hotel_comments_cleaned.csv")
print(f"加载评论数据: {len(df_cleaned)} 条")
print(f"列名: {list(df_cleaned.columns)}")

# 加载类别配置
with open(CONFIG_DIR / "categories.json", encoding="utf-8") as f:
    categories_config = json.load(f)

# 提取 14 个小类名称和关键词
subcategory_names = []
subcategory_keywords = {}  # {name: [keywords]}
for cat in categories_config["categories"]:
    for sub in cat["subcategories"]:
        subcategory_names.append(sub["name"])
        subcategory_keywords[sub["name"]] = sub.get("subcategories", [])

print(f"类别体系: {len(subcategory_names)} 个小类")
print(f"类别列表: {subcategory_names}")

# 加载 LLM 分类结果（用于对比）
df_enriched = pd.read_csv(PROCESSED_DIR / "enriched_comments.csv", index_col=0)
print(f"\n加载 LLM 分类结果: {len(df_enriched)} 条")

# 提取 LLM 分类的 categories（category1/2/3 → 数组）
def get_llm_categories(row):
    cats = []
    for col in ["category1", "category2", "category3"]:
        if col in row and pd.notna(row[col]) and row[col]:
            cats.append(row[col])
    return cats

df_enriched["llm_categories"] = df_enriched.apply(get_llm_categories, axis=1)
print(f"LLM 分类统计: 平均每评论 {df_enriched['llm_categories'].apply(len).mean():.2f} 个类别")

## 2. 文本向量化

In [ ]:
def embed_batch(texts, batch_size=20):
    """批量生成 embedding"""
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="向量化"):
        batch = texts[i:i+batch_size]
        response = TextEmbedding.call(
            api_key=API_KEY,
            model="text-embedding-v4",
            input=batch,
            dimension=1024
        )
        if response.status_code == 200:
            embeddings.extend([item["embedding"] for item in response.output["embeddings"]])
        else:
            raise RuntimeError(f"Embedding 失败: {response.message}")
    return np.array(embeddings)

# 对所有评论生成 embedding
comments = df_cleaned["comment"].fillna("").tolist()
print(f"开始向量化 {len(comments)} 条评论...")
start_time = time.time()
embeddings = embed_batch(comments)
print(f"向量化完成，耗时 {time.time() - start_time:.1f}s，shape: {embeddings.shape}")

## 3. 降维（PCA 50D）

In [ ]:
# PCA 降维到 50 维（用于聚类）
pca_50 = PCA(n_components=50, random_state=42)
embeddings_50d = pca_50.fit_transform(embeddings)
print(f"PCA 降维: {embeddings.shape} → {embeddings_50d.shape}")
print(f"累计解释方差: {pca_50.explained_variance_ratio_.sum():.3f}")

# PCA 降维到 2 维（用于可视化）
pca_2d = PCA(n_components=2, random_state=42)
embeddings_2d = pca_2d.fit_transform(embeddings)
print(f"PCA 2D 累计解释方差: {pca_2d.explained_variance_ratio_.sum():.3f}")

## 4. 聚类

In [ ]:
cluster_labels = None
cluster_method = None
n_clusters = 0

if HAS_HDBSCAN:
    # HDBSCAN 聚类
    print("使用 HDBSCAN 聚类...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=5,
        metric="euclidean",
        cluster_selection_epsilon=0.5
    )
    cluster_labels = clusterer.fit_predict(embeddings_50d)
    cluster_method = "HDBSCAN"
    
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = (cluster_labels == -1).sum()
    print(f"HDBSCAN 聚类完成: {n_clusters} 个簇, {n_noise} 个噪声点")
else:
    # KMeans 聚类（备选）
    print("使用 KMeans 聚类...")
    n_clusters = 20  # 预设簇数，后续可调
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings_50d)
    cluster_method = "KMeans"
    print(f"KMeans 聚类完成: {n_clusters} 个簇")

# 统计每个簇的大小
cluster_counts = Counter(cluster_labels)
print(f"\n簇大小分布 (Top 10):")
for label, count in cluster_counts.most_common(10):
    label_name = f"噪声" if label == -1 else f"簇{label}"
    print(f"  {label_name}: {count} 条")

## 5. 簇 → 类别映射（关键词匹配）

In [ ]:
def extract_cluster_keywords(texts, topk=15):
    """从簇中提取高频关键词（TF-IDF + jieba）"""
    if not texts:
        return []
    # 合并所有文本
    combined = " ".join(texts)
    # 使用 jieba TF-IDF 提取关键词
    keywords = jieba.analyse.extract_tags(combined, topK=topk, withWeight=True)
    return [kw for kw, _ in keywords]

# 为每个簇提取关键词
cluster_keywords = {}
for label in set(cluster_labels):
    if label == -1:
        continue
    mask = cluster_labels == label
    cluster_texts = df_cleaned.loc[mask, "comment"].fillna("").tolist()
    cluster_keywords[label] = extract_cluster_keywords(cluster_texts)

print(f"已为 {len(cluster_keywords)} 个簇提取关键词")

# 关键词 → 类别匹配
def match_keywords_to_category(cluster_kws, category_kws):
    """计算簇关键词与类别关键词的重叠度"""
    if not cluster_kws or not category_kws:
        return 0.0
    overlap = len(set(cluster_kws) & set(category_kws))
    return overlap / max(len(category_kws), 1)

# 为每个簇分配最匹配的类别
cluster_to_category = {}
for label, kws in cluster_keywords.items():
    scores = {}
    for cat_name, cat_kws in subcategory_keywords.items():
        scores[cat_name] = match_keywords_to_category(kws, cat_kws)
    # 取最高分的类别（阈值 > 0）
    best_cat = max(scores, key=scores.get)
    if scores[best_cat] > 0:
        cluster_to_category[label] = best_cat
    else:
        cluster_to_category[label] = "未分类"

print(f"\n簇 → 类别映射:")
for label, cat in sorted(cluster_to_category.items()):
    count = cluster_counts.get(label, 0)
    kws = cluster_keywords.get(label, [])
    print(f"  簇{label} → {cat} ({count}条) | 关键词: {', '.join(kws[:5])}")

## 6. 多标签分配（余弦相似度）

In [ ]:
# 构建 14 个类别的中心向量
# 策略：用每个类别的关键词拼接成描述文本，生成 embedding 作为类别中心
category_centers = {}
category_descriptions = {}

for cat_name, kws in subcategory_keywords.items():
    desc = f"酒店评论关于{cat_name}：" + "，".join(kws)
    category_descriptions[cat_name] = desc

# 批量生成类别中心 embedding
cat_names_ordered = list(subcategory_keywords.keys())
cat_descs_ordered = [category_descriptions[name] for name in cat_names_ordered]

print("生成类别中心向量...")
cat_embeddings = embed_batch(cat_descs_ordered)
for i, name in enumerate(cat_names_ordered):
    category_centers[name] = cat_embeddings[i]

print(f"已生成 {len(category_centers)} 个类别中心向量")

# 多标签分配
SIMILARITY_THRESHOLD = 0.5  # 余弦相似度阈值

def assign_multi_labels(comment_emb, threshold=SIMILARITY_THRESHOLD):
    """为单条评论分配多个类别标签"""
    labels = []
    for cat_name in cat_names_ordered:
        center = category_centers[cat_name]
        sim = cosine_similarity([comment_emb], [center])[0][0]
        if sim >= threshold:
            labels.append(cat_name)
    return labels

# 对所有评论分配多标签
print("分配多标签...")
multi_labels = []
for i in tqdm(range(len(embeddings)), desc="多标签分配"):
    labels = assign_multi_labels(embeddings[i])
    multi_labels.append(labels)

# 统计
label_counts = [len(l) for l in multi_labels]
print(f"\n多标签分配统计:")
print(f"  平均每评论: {np.mean(label_counts):.2f} 个类别")
print(f"  中位数: {np.median(label_counts):.0f} 个类别")
print(f"  无标签: {sum(1 for l in multi_labels if len(l) == 0)} 条")
print(f"  最多标签: {max(label_counts)} 个")

# 类别分布
all_labels_flat = [l for labels in multi_labels for l in labels]
label_dist = Counter(all_labels_flat)
print(f"\n类别分布:")
for cat, count in label_dist.most_common():
    print(f"  {cat}: {count} 条")

## 7. 与 LLM 分类结果对比

In [ ]:
from sklearn.metrics import cohen_kappa_score

# 对比策略：对于每条评论，比较聚类分类和 LLM 分类的标签集合
# 使用 Jaccard 相似度衡量一致性

def jaccard_similarity(set1, set2):
    """计算两个集合的 Jaccard 相似度"""
    if not set1 and not set2:
        return 1.0
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)

# 确保索引对齐
common_ids = df_cleaned.index.intersection(df_enriched.index)
print(f"可对比评论数: {len(common_ids)}")

jaccard_scores = []
for idx in common_ids:
    llm_cats = set(df_enriched.loc[idx, "llm_categories"])
    cluster_cats = set(multi_labels[idx])
    jaccard_scores.append(jaccard_similarity(llm_cats, cluster_cats))

avg_jaccard = np.mean(jaccard_scores)
print(f"\n平均 Jaccard 相似度: {avg_jaccard:.4f}")
print(f"中位数 Jaccard: {np.median(jaccard_scores):.4f}")

# 按评论的 LLM 类别数分组分析
df_enriched_aligned = df_enriched.loc[common_ids]
for n_cats in range(1, 4):
    mask = df_enriched_aligned["llm_categories"].apply(len) == n_cats
    if mask.sum() > 0:
        scores_subset = [jaccard_scores[i] for i, m in enumerate(mask) if m]
        print(f"  LLM 分配 {n_cats} 个类别的评论 (n={mask.sum()}): Jaccard = {np.mean(scores_subset):.4f}")

# 类别级别的一致性
print(f"\n各类别在两种方法中的覆盖率:")
for cat_name in subcategory_names:
    llm_count = sum(1 for idx in common_ids if cat_name in df_enriched.loc[idx, "llm_categories"])
    cluster_count = sum(1 for idx in common_ids if cat_name in multi_labels[idx])
    print(f"  {cat_name}: LLM={llm_count}, 聚类={cluster_count}")

## 8. 保存聚类分类结果

In [ ]:
# 将聚类分类结果保存到 enriched_comments.csv
# 格式：categories 列存储 JSON 数组

df_result = df_cleaned.copy()
df_result["categories"] = [json.dumps(labels, ensure_ascii=False) for labels in multi_labels]
df_result["category_count"] = [len(l) for l in multi_labels]

# 也保留 cluster_label 和 cluster_category 供参考
df_result["cluster_label"] = cluster_labels
df_result["cluster_category"] = [cluster_to_category.get(l, "噪声") for l in cluster_labels]

# 保存
output_path = PROCESSED_DIR / "enriched_comments_clustered.csv"
df_result.to_csv(output_path, index=True)
print(f"聚类分类结果已保存: {output_path}")
print(f"共 {len(df_result)} 条评论")
print(f"类别分布: {df_result['category_count'].describe()}")

# 同时保存聚类模型和类别中心（供后续使用）
import pickle

clustering_artifacts = {
    "cluster_labels": cluster_labels,
    "cluster_to_category": cluster_to_category,
    "cluster_keywords": cluster_keywords,
    "category_centers": {k: v.tolist() for k, v in category_centers.items()},
    "category_names": cat_names_ordered,
    "pca_50": pca_50,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "method": cluster_method
}

with open(PROCESSED_DIR / "clustering_artifacts.pkl", "wb") as f:
    pickle.dump(clustering_artifacts, f)
print(f"聚类模型已保存: {PROCESSED_DIR / 'clustering_artifacts.pkl'}")

## 9. 可视化

In [ ]:
import matplotlib.pyplot as plt

# 2D 可视化聚类结果
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图：按簇着色
ax = axes[0]
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=cluster_labels, cmap="tab20", alpha=0.6, s=10)
ax.set_title(f"{cluster_method} 聚类结果 ({n_clusters} 个簇)")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
plt.colorbar(scatter, ax=ax, label="簇标签")

# 右图：按类别数量着色
ax2 = axes[1]
scatter2 = ax2.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                       c=[len(l) for l in multi_labels], cmap="YlOrRd", alpha=0.6, s=10)
ax2.set_title("多标签数量分布")
ax2.set_xlabel("PCA 1")
ax2.set_ylabel("PCA 2")
plt.colorbar(scatter2, ax=ax2, label="类别数")

plt.tight_layout()
plt.savefig(DATA_DIR / "evaluation" / "fig_clustering_overview.png", dpi=150, bbox_inches="tight")
plt.show()

# 类别分布对比图
fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(subcategory_names))
width = 0.35

llm_counts = [sum(1 for idx in common_ids if cat_name in df_enriched.loc[idx, "llm_categories"]) for cat_name in subcategory_names]
cluster_counts_list = [sum(1 for idx in common_ids if cat_name in multi_labels[idx]) for cat_name in subcategory_names]

ax.bar([i - width/2 for i in x], llm_counts, width, label="LLM 分类", alpha=0.8)
ax.bar([i + width/2 for i in x], cluster_counts_list, width, label="聚类分类", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(subcategory_names, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("评论数")
ax.set_title("LLM 分类 vs 聚类分类 — 类别分布对比")
ax.legend()

plt.tight_layout()
plt.savefig(DATA_DIR / "evaluation" / "fig_category_distribution_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 总结

本 Notebook 完成了以下工作：
1. 使用 text-embedding-v4 对所有评论生成 1024 维向量
2. PCA 降维到 50 维后进行 HDBSCAN/KMeans 聚类
3. 通过关键词匹配建立簇 → 类别映射
4. 基于余弦相似度实现多标签分配（阈值 0.5）
5. 与 LLM 分类结果对比，计算 Jaccard 相似度
6. 保存聚类分类结果和模型文件